# L3 calendar summaries and historical anomalies

Run all cells in Google Colab after L1. This notebook reads the three L1 calendar tables and writes all ten L3 tables and `_log_lote3.csv`. Push the L3 source before running from the remote repository.

Historical references require complete periods within the configured coverage limit. Anomalies exclude the evaluated year. MODIS gains Aqua in 2002 and the combined series gains VIIRS in 2018, introducing changes in observation coverage.

For private repositories, enable a `GITHUB_TOKEN` Colab secret.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'],
                   env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import subprocess
import pandas as pd
from metrics import calendar_summary, history_anomaly

importlib.reload(calendar_summary)
importlib.reload(history_anomaly)
METRICS = ROOT / 'metrics'
REF_START = None  # Inclusive year, or None for no lower bound.
REF_END = None
MIN_REF_YEARS = 5
MAX_UNKNOWN_DAYS = 0
START = None  # Inclusive (year, month) selection for CAL-05.
END = None
PARAMETERS = dict(ref_start=REF_START, ref_end=REF_END,
                  min_ref_years=MIN_REF_YEARS, max_unknown_days=MAX_UNKNOWN_DAYS,
                  start=START, end=END)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
INPUT_COLUMNS = {
    'l1_cal_monthly': ['country', 'year', 'month', 'series', 'cell_days', 'partial', 'n_unknown_days'],
    'l1_cal_annual': ['country', 'year', 'series', 'cell_days', 'partial', 'n_unknown_days'],
    'l1_cal_daily': ['country', 'date', 'series', 'n_cells'],
}


In [ ]:
missing = [str(METRICS / f'{name}.parquet') for name in INPUT_COLUMNS
           if not (METRICS / f'{name}.parquet').is_file()]
if missing:
    raise FileNotFoundError('Missing L1 input tables. Run notebook 02_metrics_lote1.ipynb first: '
                            + ', '.join(missing))
inputs = {name: pd.read_parquet(METRICS / f'{name}.parquet', columns=columns)
          for name, columns in INPUT_COLUMNS.items()}
monthly = inputs['l1_cal_monthly']
annual = inputs['l1_cal_annual']
daily = inputs['l1_cal_daily']
reference_options = dict(ref_start=REF_START, ref_end=REF_END,
                         min_ref_years=MIN_REF_YEARS, max_unknown_days=MAX_UNKNOWN_DAYS)
cal09 = calendar_summary.cal09_monthly_mean(monthly, **reference_options)
his03 = history_anomaly.his03_anomaly_abs(monthly, **reference_options)
tables = {
    'l3_cal05_total_period': calendar_summary.cal05_total_period(monthly, start=START, end=END),
    'l3_cal06_peak_month': calendar_summary.cal06_peak_month(monthly, max_unknown_days=MAX_UNKNOWN_DAYS),
    'l3_cal07_peak_year': calendar_summary.cal07_peak_year(annual, max_unknown_days=MAX_UNKNOWN_DAYS),
    'l3_cal08_annual_mean': calendar_summary.cal08_annual_mean(annual, ref_start=REF_START,
                                                            ref_end=REF_END, max_unknown_days=MAX_UNKNOWN_DAYS),
    'l3_cal09_monthly_mean': cal09,
    'l3_cal10_active_days': calendar_summary.cal10_active_days(daily, monthly),
    'l3_cal11_peak_day': calendar_summary.cal11_peak_day(daily),
    'l3_cal12_seasonal_profile': calendar_summary.cal12_seasonal_profile(cal09),
    'l3_his03_anomaly_abs': his03,
    'l3_his04_anomaly_pct': history_anomaly.his04_anomaly_pct(his03),
}


In [ ]:
log = []
executed_at = datetime.now(timezone.utc).isoformat()
for name, result in tables.items():
    path = METRICS / f'{name}.parquet'
    temporary = path.with_suffix('.parquet.tmp')
    result.to_parquet(temporary, index=False)
    temporary.replace(path)
    log.append(dict(metric=name, parameters=json.dumps(PARAMETERS, sort_keys=True),
                    commit=COMMIT, output_rows=len(result), executed_at_utc=executed_at))
log_path = METRICS / '_log_lote3.csv'
temporary = log_path.with_suffix('.csv.tmp')
pd.DataFrame(log).to_csv(temporary, index=False)
temporary.replace(log_path)


In [ ]:
active = tables['l3_cal10_active_days']
assert active.active_days.le(active.n_days_in_month).all()
profile = tables['l3_cal12_seasonal_profile']
shares = profile.groupby(['country', 'series'], observed=True, dropna=False).share_pct.sum(min_count=1).dropna()
assert shares.sub(100).abs().lt(1e-9).all()
assert tables['l3_his03_anomaly_abs'].shape[0] == monthly.shape[0]
for name, result in tables.items():
    print(name, result.shape)
    print(result.dtypes)
    print(result.head(3).to_string(index=False))
print(f'L3 completed: {len(tables)} tables written to {METRICS}')
